# Computational Analysis of Sounds and Music (CH-CASM-M)

## 09 - Music Transcription 1

**WS 2025/2026**

Prof. Dr. Jakob Abeßer, jakob.abesser@uni-bamberg.de

Last update: 17.12.2025

**Outline**

In this notebook, you will learn how to implement a simple **melody transcription** algorithm using a **CNN model**.

In [ ]:
!pip install wget

In [ ]:
import glob
import os
import librosa
import numpy as np
import matplotlib.pyplot as pl
import numpy as np
import IPython.display as ipd
import wget
import zipfile
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D

## Dataset

We need a small dataset of audio files with corresponding pitch annotations.

We use 4 files of the **MDB-melody-synth** dataset (http://synthdatasets.weebly.com/mdb-melody-synth.html), which "contains 65 songs from the MedleyDB dataset in which the melody track has been resynthesized to obtain a perfect melody f0 annotation".

In [ ]:
if not os.path.isfile('MDB-melody-synth.zip'):
    print('Please wait a couple of seconds ...')
    wget.download('https://github.com/CHBamberg/CH-CASM-M-2025/raw/refs/heads/main/data/MDB-melody-synth.zip', 
                      out='MDB-melody-synth.zip', bar=None)
    print('MDB-melody-synth downloaded successfully ...')
else:
    print('Files already exist!')
    
if not os.path.isdir('MDB-melody-synth.zip'):
    print("Let's unzip the file ... ")
    assert os.path.isfile('MDB-melody-synth.zip')
    with zipfile.ZipFile('MDB-melody-synth.zip', 'r') as f:
        # Entpacke alle Inhalte in das angegebene Verzeichnis
        f.extractall('.')
    assert os.path.isdir('MDB-melody-synth')
    print("All done :)")

dir_dataset = 'MDB-melody-synth'

Let's check our dataset:

In [ ]:
# List files in the directory
files = os.listdir(dir_dataset)

# Print the list of files
print(files)

We have **4 audio files (WAV)** and **4 CSV files** with the corresponding pitch annotation.

Let's just memorize the 4 file prefixes:

In [ ]:
file_prefixes = list(set([_[:-4] for _ in files]))
print(file_prefixes)

Let's listen to the first file...

In [ ]:
fn_wav = os.path.join(dir_dataset, file_prefixes[0] + '.wav')
x, fs = librosa.load(fn_wav)
ipd.display(ipd.Audio(data=x, rate=fs))

**Observation**: This recording sounds like female singing, but is actually a resythesized version of a transcribed female vocal track, which causes some audible artifacts.

## Feature Extraction

We need a function to compute the **Constant-Q transform** from all these files. 
We use the following parameters (inspired by https://arxiv.org/abs/1810.12947)

- minimum frequency (31 Hz = note B0)
- number of frequebcy bins = 320 (upper frequency: 1250 Hz (D#6))
- frequency resolution (60 bins per octave)


In [ ]:
def compute_cqt(fn_wav, fs=44100, hop_length=256, bpo=5*12, fmin=31, nbins = 320):
    """
    Compute the Constant-Q transform (CQT) with librosa
    Args:
        fn_wav (str): File path to the input audio file (in .wav format).
        fs (int, optional): Sampling rate of the audio file (in Hz). Default is 22050 Hz.
        hop_length (int, optional): Number of samples between successive CQT columns. Default is 256.
        bpo (int, optional): Number of bins per octave for the CQT. Default is 5*12.
        fmin (int, optional): Minimum frequency (in Hz) of the CQT. Default is 31 Hz.
        nbins (int, optional): Number of bins in the CQT. Default is 320.

    Returns:
        cqt (np.ndarray): 2D array containing the magnitude CQT coefficients.
        time_axis (np.ndarray): Numpy array with frame times in seconds.
        freq_hz (np.ndarray): Frequency values in Hz of all frequency bands of the CQT.
    """
    x, fs = librosa.load(fn_wav, sr=fs, mono=True)
    cqt = np.abs(librosa.core.cqt(x, sr=fs, 
                                  hop_length=hop_length, 
                                  fmin=fmin,
                                  bins_per_octave=bpo,
                                  n_bins=nbins))
    cqt = librosa.amplitude_to_db(cqt, ref=np.max)
    cqt = cqt.astype(np.float16)
    time_axis = np.arange(cqt.shape[1])*hop_length/fs
    freq_hz = librosa.cqt_frequencies(n_bins=nbins, fmin=fmin, bins_per_octave=bpo)

    return cqt, time_axis, freq_hz
    

In [ ]:
# let's try it:
cqt, cqt_times_sec, cqt_freq_hz = compute_cqt(fn_wav)


# plot the first part of the CQT
pl.figure(figsize=(10,3))
librosa.display.specshow(cqt, sr=44100, x_axis='time', y_axis='cqt_note', ax=pl.gca())
pl.show()

## Import pitch annotations

In [ ]:
# let's inspect the first CSV file
fn_csv = os.path.join(dir_dataset, file_prefixes[0] + '.csv')
data = np.loadtxt(fn_csv, delimiter=',')
print(data[8650:8670, :])

pl.figure()
pl.plot(data[:,1])
pl.show()

**Observation**: 

The first column includes **time-stamps in seconds** and the second column **fundamental frequency values of the singing voice in Hz**.
**Unvoiced** frames have no fundamental frequency and are labeled with "0".

## Annotation Mapping

**Problem**: 

We have a) our pitch annotations and b) the time axis of our CQT, which are **not matching**.

**Goal**

We want a binary matrix of the same shape as our **CQT** matrix, which has **ones** wherever a pitched frame is annotated.

In [ ]:
def create_cqt_target(cqt_times_sec, cqt_freq_hz, pitch_times_sec, pitch_freq_hz):
    n_frames = len(cqt_times_sec)
    n_bins = len(cqt_freq_hz)
    
    cqt_target = np.zeros((n_bins, n_frames), dtype=bool)
    n_pitch_frames = len(pitch_freq_hz)
    for i in range(n_pitch_frames):
        # only proceed if frame is not unvoiced
        if pitch_freq_hz[i] > 0:
            # find closest frequency bin
            cqt_freq_bin = np.argmin(np.abs(cqt_freq_hz-pitch_freq_hz[i]))
            # find closest time frame
            cqt_time_frame = np.argmin(np.abs(cqt_times_sec-pitch_times_sec[i]))
            # "mark" it as "True"
            cqt_target[cqt_freq_bin, cqt_time_frame] = True
            
    return cqt_target

In [ ]:
# let's try this again for the first song
fn_wav = os.path.join(dir_dataset, file_prefixes[0] + '.wav')
cqt, cqt_times_sec, cqt_freq_hz = compute_cqt(fn_wav)

# load annotation file
fn_csv = os.path.join(dir_dataset, file_prefixes[0] + '.csv')
data = np.loadtxt(fn_csv, delimiter=',')
pitch_times_sec = data[:, 0]
pitch_freq_hz = data[:, 1]

# create targets for CQT matrix
cqt_target = create_cqt_target(cqt_times_sec, cqt_freq_hz, pitch_times_sec, pitch_freq_hz)

# finally, let's plot it 
pl.figure(figsize=(15,5))
pl.subplot(1,2,1)
librosa.display.specshow(cqt[:, :], sr=44100, x_axis='time', y_axis='cqt_note', ax=pl.gca())
pl.title('CQT')
pl.subplot(1,2,2)
pl.imshow(cqt_target[:, :], aspect="auto", origin="lower", cmap="Greys")
pl.title('Target')
pl.tight_layout()
pl.show()


**Observation**

The **piano-roll** target on the right shows the fundamental frequency contour. This is what we want our model to predict.

## Batch Feature Extraction

Nice, so let's compute all 4 CQT spectrograms and all 4 target matrices: (**this takes some seconds ...**)

In [ ]:
all_cqts = []
all_targets = []

for i in range(4):
    print(f"Process file {i+1}/4")
    # (1) compute CQT
    fn_wav = os.path.join(dir_dataset, file_prefixes[i] + '.wav')
    cqt, cqt_times_sec, cqt_freq_hz = compute_cqt(fn_wav)
    
    # (2) compute target matrix
    fn_csv = os.path.join(dir_dataset, file_prefixes[0] + '.csv')
    data = np.loadtxt(fn_csv, delimiter=',')
    pitch_times_sec = data[:, 0]
    pitch_freq_hz = data[:, 1]
    cqt_target = create_cqt_target(cqt_times_sec, cqt_freq_hz, pitch_times_sec, pitch_freq_hz)
    
    all_cqts.append(cqt)
    all_targets.append(cqt_target)

# we'll use the first three files as training and the final file as test set
all_cqts_train = np.hstack(all_cqts[:3])
all_targets_train = np.hstack(all_targets[:3])

all_cqts_test = all_cqts[3]
all_targets_test = all_targets[3]

print(f"Final shape (training) = {all_cqts_train.shape}")
print(f"Final shape (test) = {all_cqts_test.shape}")

    

## Neural Network

### Patches

For simplicity, well cut our matrices into patches of a fixed size (250 frames) with no overlap.

In [ ]:
feat_train = []
target_train = []
n_frames = all_cqts_train.shape[1]
offset = 0
patch_len = 250
patch_hop = 250

while True:
    if offset + patch_len >= n_frames:
        break
    feat_train.append(all_cqts_train[:, offset: offset + patch_len])
    target_train.append(all_targets_train[:, offset: offset + patch_len])
    offset += patch_hop
feat_train = np.stack(feat_train)
target_train = np.stack(target_train)

# finally, we'll add a singleton dimension (one channel)
feat_train = np.expand_dims(feat_train, -1)
target_train = np.expand_dims(target_train, -1)

print(f"Feature & target array shape = {feat_train.shape}")



## Neural Network Architecture

We're using the **functional API** (https://keras.io/guides/functional_api/) to set up a simple CNN model **without any pooling operation** (since our input has the same shape as our output).

In [ ]:

def create_fully_convolutional_model(input_shape):
    # Define input layer
    inputs = Input(shape=input_shape)
    
    # First convolutional layer
    conv1 = Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same')(inputs)
    
    # Second convolutional layer
    conv2 = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(conv1)
    
    # Third convolutional layer (final layer with sigmoid activation)
    outputs = Conv2D(1, kernel_size=(3, 3), activation='sigmoid', padding='same')(conv2)
    
    # Create model
    model = Model(inputs=inputs, outputs=outputs)
    
    model.compile(loss='binary_crossentropy', 
              optimizer='adam')
 
    return model

# Example usage
input_shape = feat_train.shape[1:] 
model = create_fully_convolutional_model(input_shape)
model.summary()



## Model Training

In [ ]:
history = model.fit(feat_train, target_train, epochs=5, batch_size=64, verbose=1)

In [ ]:
pl.plot(history.history['loss'])
pl.ylabel('Loss')
pl.xlabel('Epoch')
pl.show()

## Model Test

We're testing our model by making predictions on the fourth file.
The approach is to 
 - cut in into similarly sized patches 
 - compute prediction for each patch
 - concatenate the predictions

In [ ]:
print(all_cqts_test.shape)

In [ ]:
all_pred = []
offset = 0
n_frames = all_cqts_test.shape[1]
while True:
    patch = all_cqts_test[:, offset:offset+patch_len]
    patch = np.expand_dims(patch, 0)
    patch = np.expand_dims(patch, -1)
    
    # compute model prediction for current 
    curr_pred = model.predict(patch)
    # reduce to 2D piano roll (pitch vs. time)
    curr_pred = curr_pred[0, :, :, 0]
    all_pred.append(curr_pred)
    
    offset += patch_len
    
    if offset > n_frames-patch_len:
        break
        
# finally, let's combine all patch predictions into one piano roll
all_pred = np.hstack(all_pred)

print(f"Final shape: {all_pred.shape}")

## Result Visualization

In [ ]:
print(all_targets_test.shape)

pl.figure(figsize=(15,3))
pl.subplot(1,2,1)
librosa.display.specshow(all_cqts_test[:, 10000:15000], sr=44100, x_axis='time', y_axis='cqt_note', ax=pl.gca())
pl.title('CQT')
pl.subplot(1,2,2)
pl.imshow(all_pred[:, 10000:15000], aspect="auto", origin="lower",cmap="Greys")
pl.title('Predicted Pitch Contour')
pl.show()

## Results

The predicted pitch contour looks quite good given the 
  - small amount of training data
  - few training epochs
that we used.

**But**: There are still several harmonics visible, so the model has not yet learnt to suppress them fully.

## Possible next steps

- change modeling approach:
  - use a multi-class classification (with softmax function in the final layer and categorical crossentropy as loss function) to predict always one of the frames as pitch frames
  - for this we need to add an 
- use larger part of the MDB-Stem-Synth dataset
- try data augmentation
- train for more epochs